# Assignment 2: Incremental Ingestion with Apache Spark
# Victoria Essien

In [1]:
import os

# setting the correct java version for spark to use
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home'
os.environ['PYSPARK_PYTHON'] = 'python3'

print("Java path set!")

Java path set!


#### Starting spark session

In [4]:
from pyspark.sql import SparkSession

# starting spark session
spark = SparkSession.builder \
    .appName("assignment2") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print("Spark is ready!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/01 15:57:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Spark is ready!


#### Generating two sales files for the assignment 2

In [5]:
import subprocess
import os

# creating the first sales file
subprocess.run(['python', 'generate_data.py', '--scale', '1'])
os.rename('data_1.csv.gz', 'file_1.csv.gz')
print("file_1 is ready")

# creating the second sales file
subprocess.run(['python', 'generate_data.py', '--scale', '1'])
os.rename('data_1.csv.gz', 'file_2.csv.gz')
print("file_2 is ready")

Generating dataset with scale: 1
Generated data file.
file_1 is ready
Generating dataset with scale: 1
Generated data file.
file_2 is ready


#### Adding duplicate rows to the second sales file

In [8]:
import gzip

# read the first 20 rows from file_1
rows_from_file1 = []
with gzip.open('file_1.csv.gz', 'rt') as f:
    for i, line in enumerate(f):
        rows_from_file1.append(line)
        if i >= 19:
            break

# adding those rows to file_2 to create duplicates data
with gzip.open('file_2.csv.gz', 'at') as f:
    for row in rows_from_file1:
        f.write(row)

print(f"Added {len(rows_from_file1)} duplicate rows to file_2")
print("file_2 now contains duplicates from file_1")

Added 20 duplicate rows to file_2
file_2 now contains duplicates from file_1


#### Loading file_1 into spark

In [11]:
# loading file_1 into spark
sales_file1 = spark.read \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .csv("file_1.csv.gz")

# giving the columns proper names
sales_file1 = sales_file1.toDF("basket_id", "product_id")

# checking the first 5 rows
print("First 5 rows of file_1:")
sales_file1.show(5)

print(f"Total rows in file_1: {sales_file1.count()}")

First 5 rows of file_1:
+--------------------+----------+
|           basket_id|product_id|
+--------------------+----------+
|fea4f554-1220-44b...|        99|
|fea4f554-1220-44b...|       222|
|fea4f554-1220-44b...|        68|
|fea4f554-1220-44b...|       221|
|fea4f554-1220-44b...|       171|
+--------------------+----------+
only showing top 5 rows
Total rows in file_1: 196427


#### Loading file_2 into spark

In [14]:
# loading file_2 into spark
sales_file2 = spark.read \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .csv("file_2.csv.gz")

# giving the columns proper names
sales_file2 = sales_file2.toDF("basket_id", "product_id")

# checking the first 5 rows
print("First 5 rows of file_2:")
sales_file2.show(5)

print(f"Total rows in file_2: {sales_file2.count()}")

First 5 rows of file_2:
+--------------------+----------+
|           basket_id|product_id|
+--------------------+----------+
|a635ddd1-acec-444...|        76|
|a635ddd1-acec-444...|       137|
|a635ddd1-acec-444...|        15|
|a635ddd1-acec-444...|        71|
|408969db-fd72-4ac...|       199|
+--------------------+----------+
only showing top 5 rows
Total rows in file_2: 196248


#### Combining both files and also remove duplicated rows

In [17]:
# combining file_1 and file_2 into one dataframe
combined = sales_file1.union(sales_file2)
print(f"Total rows before deduplication: {combined.count()}")

# removing duplicate rows based on basket_id and product_id
deduplicated = combined.dropDuplicates(["basket_id", "product_id"])
print(f"Total rows after deduplication: {deduplicated.count()}")

Total rows before deduplication: 392675


Total rows after deduplication: 392635


#### Saving the deduplicated result as Parquet

In [20]:
# parquet is a compressed efficient format designed for big data like this records

deduplicated.write \
    .mode("overwrite") \
    .parquet("output/deduplicated_data")

print("Deduplicated data saved to output/deduplicated_data")

26/04/01 16:04:29 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


Deduplicated data saved to output/deduplicated_data


#### Validating the results 

In [25]:
# loading the saved parquet file to verify it looks correct

result = spark.read.parquet("output/deduplicated_data")

print("Final result:")
result.show(20)

print(f"Total unique records: {result.count()}")

Final result:
+--------------------+----------+
|           basket_id|product_id|
+--------------------+----------+
|a33c624b-886b-4bd...|        68|
|74ade28c-4efc-476...|       177|
|d24f8a37-1036-4e9...|        87|
|8df958ad-74d4-41e...|       221|
|44bf41bd-54b2-40b...|       119|
|9e23c798-616f-475...|       123|
|f66eb2df-998a-461...|        54|
|1792bbd3-f7b8-4f4...|       209|
|9da40628-6327-478...|        87|
|8026aec9-e19b-4dd...|       203|
|9f9cca87-2ada-497...|       166|
|5607c1dc-414a-425...|       209|
|6eb329b5-182f-4fe...|       115|
|b1ce308f-ed74-4b1...|       215|
|68e27b89-ba82-44b...|        28|
|58ddc636-0519-46b...|       152|
|a87b8c1c-aa2f-457...|       107|
|dba6af92-9f35-403...|       148|
|7954f2c9-58fd-49e...|       223|
|bd24cb6d-a62e-4e2...|       168|
+--------------------+----------+
only showing top 20 rows
Total unique records: 392635


In [27]:
# checking the row counts to confirm deduplication worked

print(f"Rows in file_1:                  {sales_file1.count()}")
print(f"Rows in file_2:                  {sales_file2.count()}")
print(f"Total rows combined:             {combined.count()}")
print(f"Total rows after deduplication:  {deduplicated.count()}")
print()

# calculating how many duplicates were removed
duplicates_removed = combined.count() - deduplicated.count()
print(f"Duplicate rows removed:          {duplicates_removed}")

Rows in file_1:                  196427
Rows in file_2:                  196248
Total rows combined:             392675


Total rows after deduplication:  392635

Duplicate rows removed:          40


### A finalised summary that can run with a single command for faster comprehension

In [30]:
from pyspark.sql import SparkSession

def incremental_ingestion(input_files, output_path):
    """
    this function reads multiple sales files one by one,
    removes any duplicate basket and product combinations
    and saves the clean result as parquet
    """

    print(f"starting to process {len(input_files)} files...")
    print()

    # I will start with nothing and keep adding files one by one
    combined = None

    for file in input_files:
        print(f"loading {file}...")

        # reading the file into spark
        df = spark.read \
            .option("header", "false") \
            .option("inferSchema", "true") \
            .csv(file)

        # the file has no headers so I am adding them manually
        df = df.toDF("basket_id", "product_id")

        # adding this file to the combined data
        if combined is None:
            combined = df
        else:
            combined = combined.union(df)

    # checking how many rows we have before cleaning
    total_before = combined.count()
    print()
    print(f"total rows before removing duplicates: {total_before}")

    # removing rows where basket_id and product_id are the same
    clean_data = combined.dropDuplicates(["basket_id", "product_id"])

    # checking how many rows we have after cleaning
    total_after = clean_data.count()
    print(f"total rows after removing duplicates:  {total_after}")
    print(f"duplicates removed:                    {total_before - total_after}")

    # saving the clean data as parquet
    # parquet is a compressed format that is faster and smaller than csv
    clean_data.write \
        .mode("overwrite") \
        .parquet(output_path)

    print()
    print(f"clean data saved to {output_path}")

    return clean_data


# running the ingestion with both files
result = incremental_ingestion(
    input_files=['file_1.csv.gz', 'file_2.csv.gz'],
    output_path='output/deduplicated_data'
)

starting to process 2 files...

loading file_1.csv.gz...
loading file_2.csv.gz...

total rows before removing duplicates: 392675
total rows after removing duplicates:  392635
duplicates removed:                    40


26/04/01 16:12:00 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers



clean data saved to output/deduplicated_data
